# Exercise 1 · Write the path to each value in a return

You write where three values sit, and what may be missing.

**Goal:** Write the path to three values in a return, and find the elements that may be missing or repeat.

**Why it matters:** Code finds a value by its path, namespace included. One return cannot show what another lacks.

**Before you start:** You can start Codex (Monday's exercises).

**What the check measures:** each value in `paths.json` is right, each path runs, and you committed the file.

**Done when:** the last cell prints `Done:`.

## The situation

`returns/` holds three returns in the IRS e-file shape. `try_paths.py` runs the paths you write in `paths.json`. The files are made up.

## Step 1 · Write the practice files

This is the solved copy. Run the cell. It writes `practice/` as it is after Step 7: the filled `paths.json`, and two commits. It never overwrites a file.

Expect six `Wrote practice/` lines and `Committed the start and the paths`.

In [ ]:
FILES = {
    "returns/return-a.xml": """<!-- Every value in this file is made up. -->
<Return xmlns="http://www.irs.gov/efile">
  <ReturnHeader>
    <PrimarySSN>400001234</PrimarySSN>
  </ReturnHeader>
  <ReturnData>
    <IRS1040 documentId="DOC0001">
      <WagesAmt>41000</WagesAmt>
      <DependentDetail>
        <DependentFirstNm>TEST CHILD ONE</DependentFirstNm>
      </DependentDetail>
      <DependentDetail>
        <DependentFirstNm>TEST CHILD TWO</DependentFirstNm>
      </DependentDetail>
    </IRS1040>
  </ReturnData>
</Return>
""",
    "returns/return-b.xml": """<!-- Every value in this file is made up. -->
<Return xmlns="http://www.irs.gov/efile">
  <ReturnHeader>
    <PrimarySSN>400005678</PrimarySSN>
  </ReturnHeader>
  <ReturnData>
    <IRS1040 documentId="DOC0001">
      <WagesAmt>52000</WagesAmt>
    </IRS1040>
  </ReturnData>
</Return>
""",
    "returns/return-c.xml": """<!-- Every value in this file is made up. -->
<Return xmlns="http://www.irs.gov/efile">
  <ReturnHeader>
    <PrimarySSN>400009876</PrimarySSN>
  </ReturnHeader>
  <ReturnData>
    <IRS1040 documentId="DOC0001">
      <DependentDetail>
        <DependentFirstNm>TEST CHILD ONE</DependentFirstNm>
      </DependentDetail>
    </IRS1040>
  </ReturnData>
</Return>
""",
    "try_paths.py": """import json
import xml.etree.ElementTree as ET

NS = {"efile": "http://www.irs.gov/efile"}


def find_all(path: str, xml_path: str) -> list[str]:
    \"\"\"Find what a path reaches in one return.

    Args:
        path: A path that starts below the root.
        xml_path: The return's file.

    Returns:
        Each element's text, or its attributes if it has no text, in document order.
    \"\"\"
    root = ET.parse(xml_path).getroot()
    return [(element.text or "").strip() or str(element.attrib) for element in root.findall(path, NS)]


def main() -> None:
    \"\"\"Print each path in paths.json and what it finds in each return.\"\"\"
    with open("paths.json", encoding="utf-8") as file:
        paths = json.load(file)["paths"]
    for name, path in paths.items():
        print(f"{name}: {path or '(no path yet)'}")
        if path:
            for letter in "abc":
                print(f"  return-{letter}.xml: {find_all(path, f'returns/return-{letter}.xml')}")


if __name__ == "__main__":
    main()
""",
    "paths.json": """{
  "parts": {"who_files": "ReturnHeader", "what_they_reported": "ReturnData", "form": "IRS1040"},
  "paths": {
    "wages": "efile:ReturnData/efile:IRS1040/efile:WagesAmt",
    "ssn": "efile:ReturnHeader/efile:PrimarySSN",
    "dependent_names": "efile:ReturnData/efile:IRS1040/efile:DependentDetail/efile:DependentFirstNm"
  },
  "optional": ["WagesAmt", "DependentDetail"],
  "repeated": ["DependentDetail"]
}
""",
    ".gitignore": """__pycache__/
""",
}

from pathlib import Path

PRACTICE = Path("practice")
for name, text in FILES.items():
    path = PRACTICE / name
    if path.exists():
        print(f"Kept practice/{name}: it is already there.")
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text, encoding="utf-8")
        print(f"Wrote practice/{name}")

# The start, then the paths a learner writes in Step 5 (simulated), each committed.
import subprocess

EMPTY_PATHS = """{
  "parts": {"who_files": "", "what_they_reported": "", "form": ""},
  "paths": {"wages": "", "ssn": "", "dependent_names": ""},
  "optional": [],
  "repeated": []
}
"""
if not (PRACTICE / ".git").exists():
    (PRACTICE / "paths.json").write_text(EMPTY_PATHS, encoding="utf-8")
    for step in (["init", "-q", "-b", "main"], ["add", "-A"], ["commit", "-qm", "Start"]):
        subprocess.run(["git", "-C", str(PRACTICE), *step], check=True)
    (PRACTICE / "paths.json").write_text(FILES["paths.json"], encoding="utf-8")
    subprocess.run(["git", "-C", str(PRACTICE), "commit", "-qam", "Write the paths"], check=True)
    print("Committed the start and the paths")

## Step 2 · Open a terminal and commit the start

Choose **Terminal > New Terminal**. Type each line, then press Enter:

```sh
source .venv/bin/activate
cd week-02/day-04/exercise-01-paths-in-a-return/practice
git init -q -b main && git add -A && git commit -qm "Start"
```

Expect no output from the last line.

## Step 3 · Ask Codex for the element tree

```sh
codex --no-daemon
```

If Codex stops with an error about `--no-daemon`, run `codex` without it. If Codex asks "Trust this folder?", choose `1. Trust and continue`. Type this request:

```text
Print the element tree of returns/return-a.xml as an indented outline, with each element's full name, attributes and text. Change no file.
```

Type `/exit`. If Codex did not print it, run this line:

```sh
python3 -c "import xml.etree.ElementTree as ET; f = lambda e, d=0: [print('  ' * d + e.tag, e.attrib or '', (e.text or '').strip())] + [f(c, d + 1) for c in e]; f(ET.parse('returns/return-a.xml').getroot())"
```

Expect ten elements, each name starting with `{http://www.irs.gov/efile}`.

### Learn: a name includes its namespace

"A default namespace declaration applies to all unprefixed element names within its scope" ([Namespaces in XML](https://www.w3.org/TR/xml-names/)). So a search for `WagesAmt` alone finds nothing. Today's deck, 4.1, "A name includes its namespace: search with a prefix map.": "Search through NS, a dictionary of your own prefixes".

## Step 4 · Read the three returns

```sh
cat returns/return-b.xml returns/return-c.xml
```

Compare each with the outline of return A. Which elements does each one lack? Which element comes twice in one return?

Expect two returns like return A, each with a part missing.

## Step 5 · Fill in paths.json

Open it with `code paths.json`. Fill in each value. Save with Ctrl+S or Cmd+S:

- `parts`: the element that holds who files, what they reported, and the form.
- `paths`: the path to each value. Start below `Return`. Write `efile:` before each name, and `/` between names.
- `optional`: the outermost elements that at least one return lacks.
- `repeated`: the outermost elements that come twice or more in one return.

<details><summary>Show the answer</summary>

```json
{
  "parts": {"who_files": "ReturnHeader", "what_they_reported": "ReturnData", "form": "IRS1040"},
  "paths": {
    "wages": "efile:ReturnData/efile:IRS1040/efile:WagesAmt",
    "ssn": "efile:ReturnHeader/efile:PrimarySSN",
    "dependent_names": "efile:ReturnData/efile:IRS1040/efile:DependentDetail/efile:DependentFirstNm"
  },
  "optional": ["WagesAmt", "DependentDetail"],
  "repeated": ["DependentDetail"]
}
```

Why:

- `parts`: `ReturnHeader` holds the SSN of who files. `ReturnData` holds the forms. `IRS1040` is Form 1040 (4.1, "A return is one XML file: a header, then the return data.").
- `paths`: `try_paths.py` searches from `Return`. `DependentFirstNm` holds the name: two in return A, none in B, one in C. It goes with its parent.
- `optional`: return C lacks `WagesAmt`. Return B lacks `DependentDetail`. 4.1, "The schema says what may be missing or repeat.": "Absent: no particular meaning".
- `repeated`: return A has two `DependentDetail`.

</details>

## Step 6 · Run your paths

ElementTree reads only part of XPath, the path language. So run each path:

```sh
python3 try_paths.py
```

Expect `['41000']`, `['52000']` and `[]` under `wages`. A path that finds `[]` in all three is wrong: start it below `Return`.

## Step 7 · Commit your paths

```sh
git commit -qam "Write the paths"
```

Expect no output.

## Start over (only if you need it)

Set `START_OVER = True` and run the cell. It writes the Step 1 files again, with an empty `paths.json`. Then set it back to `False` and go to Step 5.

Expect `Nothing changed.` while it is `False`.

In [ ]:
# Start over: only if START_OVER is True, write the Step 1 files again
START_OVER = False

if not START_OVER:
    print("Nothing changed.")
elif "FILES" not in globals():
    print("Run the Step 1 cell first, then this cell.")
else:
    for name, text in FILES.items():
        (PRACTICE / name).parent.mkdir(parents=True, exist_ok=True)
        (PRACTICE / name).write_text(text, encoding="utf-8")
    print("Wrote the practice files again. Now set START_OVER back to False.")

## Check your work

Run the cell. Expect four `ok:` lines, then `Done:`. A `Not yet:` line names the first problem and what to do.

In [ ]:
# Check your work: run your paths on the three returns, then read Git
import json
import subprocess
import xml.etree.ElementTree as ET
from pathlib import Path

PRACTICE = Path("practice")
NS = {"efile": "http://www.irs.gov/efile"}
KIDS = ["TEST CHILD ONE", "TEST CHILD TWO"]
WANT = {"wages": [["41000"], ["52000"], []], "ssn": [["400001234"], ["400005678"], ["400009876"]],
        "dependent_names": [KIDS, [], KIDS[:1]]}


def git(*args):
    return subprocess.run(["git", "-C", str(PRACTICE), *args], capture_output=True, text=True).stdout


def first_problem():
    """Print ok for each check that passes; return the first problem, or ""."""
    given = json.loads((PRACTICE / "paths.json").read_text(encoding="utf-8"))
    local = lambda items: [str(item).strip().removeprefix("efile:") for item in items]
    if local(given["parts"].values()) != ["ReturnHeader", "ReturnData", "IRS1040"]:
        return "parts in paths.json is not right. Read the Step 3 outline again (Step 5)."
    print("ok: paths.json names the three parts")
    roots = [ET.parse(PRACTICE / f"returns/return-{x}.xml").getroot() for x in "abc"]
    for key, want in WANT.items():
        path = str(given["paths"][key])
        if "efile:" not in path:
            return f"paths.{key} is empty or has no efile: prefix. A name includes its namespace (Step 5)."
        if "//" in path:
            return f"paths.{key} skips elements with //. Name each one (Step 5)."
        got = [[(e.text or "").strip() for e in root.findall(path, NS)] for root in roots]
        if got != want:
            return f"paths.{key} finds {got} in returns a, b and c, not {want} (Step 5)."
    print("ok: each path finds its values in all three returns")
    optional, repeated = set(local(given["optional"])), set(local(given["repeated"]))
    if (optional - {"DependentFirstNm"} != {"WagesAmt", "DependentDetail"}
            or "DependentDetail" not in repeated or not repeated <= {"DependentDetail", "DependentFirstNm"}):
        return "optional or repeated in paths.json is not right. Compare the three returns (Step 4)."
    print("ok: optional and repeated are right")
    if not (PRACTICE / ".git").exists():
        return "practice/ has no Git repository. Do Step 2."
    if len(git("log", "--format=%s").splitlines()) < 2 or git("status", "--porcelain", "paths.json"):
        return "paths.json is not committed. Commit it (Step 7)."
    print("ok: paths.json is committed")
    return ""


try:
    problem = first_problem()
except Exception as error:
    problem = f"the check stopped on {type(error).__name__}: {error}. Run Step 1, then test your paths (Step 6)."
print(f"Not yet: {problem}" if problem else "Done: each path finds its values, and optional and repeated are right.")

## What you learned

- A path names each element below the root, each with its namespace prefix.
- Read several returns before you call an element required.

**Reflect:** Which value in your own data could be missing? How would code tell it from a zero?